# 04. Analysis

Simple statistical comparisons (t-tests) between crisis and non-crisis years, and a basic logistic regression exploring whether financing structure predicts crisis years. This is exploratory, not causal.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/processed/panel_clean.csv')
df.head()

## T-test: are ratio means different in crisis vs. non-crisis years?

In [ ]:
clean = df.dropna(subset=['banking_crisis_dummy', 'syndicated_loans_gdp', 'corporate_bonds_gdp'])
crisis = clean[clean['banking_crisis_dummy'] == 1]
no_crisis = clean[clean['banking_crisis_dummy'] == 0]

for col in ['syndicated_loans_gdp', 'corporate_bonds_gdp']:
    t, p = stats.ttest_ind(crisis[col], no_crisis[col], equal_var=False, nan_policy='omit')
    print(f'{col}: t={t:.2f}, p={p:.4f}')

## Simple logistic regression (exploratory)

Predicting the crisis dummy from the two financing ratios and GDP growth. Treat this as descriptive/exploratory — sample sizes for crisis years are small and this isn't a causal model.

In [ ]:
import statsmodels.api as sm

model_df = clean.dropna(subset=['gdp_growth_pct'])
X = model_df[['syndicated_loans_gdp', 'corporate_bonds_gdp', 'gdp_growth_pct']]
X = sm.add_constant(X)
y = model_df['banking_crisis_dummy']

logit = sm.Logit(y, X).fit()
print(logit.summary())

## Notes / caveats

- Crisis years are rare events (class imbalance) — coefficients here are descriptive, not a validated predictive model.
- Dealogic-based series (loans/bonds) have limited country coverage; results may not generalize to countries without deep capital markets.
- No lag structure applied — for a real predictive angle, lag the financing ratios by 1–2 years before the crisis indicator.